## 1. LIBRARIES:

In [ ]:
"""LIBRARIES"""
import os
import numpy as np
import matplotlib.pyplot as plt
import wavepal as wv
import spot_wave as sw

# reads SPOT_WAVE_CARMCMC_PATH, or pass path="/.../WAVEPAL/carmcmc/carma_pack/src" explicitly
sw.setup_carmcmc()


### 1.1. DATASET:

In [ ]:
main_directory = "/home/uri/juliet/Targets/starsim/PAPER/NEW_CONFIGS/ONE_ACTIVE_LONGITUDE/Prot_055.0d_medio/N0100/"
filename = main_directory + "rvs_ratio00.55_P1_030.250d.dat"

myt, myRV, myRV_err, instruments = sw.load_rv_file(filename)
unique_instruments = np.unique(instruments)

print(f"Number of points: {len(myt)}")
if len(unique_instruments) > 1:
    print(f"Detected {len(unique_instruments)} instruments: {unique_instruments}")
    for inst in unique_instruments:
        print(f"  - {inst}: N={np.sum(instruments == inst)}")
else:
    print("Only one instrument found, no mean subtraction applied.")

plt.figure(figsize=(6, 3))
if len(unique_instruments) > 1:
    for inst in unique_instruments:
        mask = instruments == inst
        plt.scatter(myt[mask], myRV[mask], s=15, label=inst)
    plt.legend()
else:
    plt.scatter(myt, myRV, s=15, color="k")
plt.xlabel("Time (days)")
plt.ylabel("RV (m/s)")
plt.tight_layout()
plt.show()

wavelet = wv.Wavepal(myt, myRV, "time", "RV", t_units="days", mydata_units="m/s")
wavelet.check_data()

plot_trend = wavelet.plot_trend(pol_degree=-1)
plot_trend


### 1.2. IS THE DATASET SUITABLE FOR A WAVELET ANALYSIS?

In [ ]:
"""EXAMPLE USAGE"""
# test_cwt_feasibility() now lives in spot_wave.utils (same logic as before,
# just imported instead of redefined in every notebook).
is_good = sw.test_cwt_feasibility(wavelet, w0=8.5)


### 1.3. CARMCMC:

In [ ]:
# CARMCMC is used in wavepal to model the correlated noise in the data and to estimate the confidence levels!
wavelet.choose_trend_degree(-1)  # Trend polynomial degree used in the analysis (0=cte, 1=linear, 2=quadratic,..., -1=no trend)
wavelet.trend_vectors()  # Trend vectors are calculated and stored in the object. Needed for CARMCMC.
# path_to_figure_folder = main_directory + "figures_carma/"
# os.makedirs(path_to_figure_folder, exist_ok=True)
# wavelet.carma_params(make_carma_fig=True, nbins=20, dpi=400, path_to_figure_folder=path_to_figure_folder)


## 2. SCALOGRAM:

In [ ]:
# spot_wave.scalogram already ships the same defaults that used to be defined
# by hand in this cell (DEFAULT_PERCENTILES / DEFAULT_TIME_STRING /
# DEFAULT_PERIOD_STRING). Override them here if this target needs different
# values:
percentile = (95., 99., 99.9)
min_period_analysis = 1.    # Minimum period analyzed in the scalogram
max_period_analysis = 200.  # Maximum period analyzed in the scalogram

# From the filename (P1_030.250d): mark the planet's candidate period as a
# reference line on the scalogram.
p_planet = 30.25


In [ ]:
# analyze_and_plot() replaces the old 3-cell sequence
# (timefreq_analysis + plot_scalogram_custom, using the time_string /
# period_string / dashed_periods that used to be typed by hand here).
fig = sw.scalogram.analyze_and_plot(
    wavelet,
    w0=7.,
    permin=min_period_analysis,
    permax=max_period_analysis,
    percentile=percentile,
    planet_periods=p_planet,
)
# fig.savefig("/home/uri/juliet/Targets/starsim/PAPER/PLANETS/ONE_ACTIVE_LONG/scalogram_rv.pdf")


### 2.1. LOOP $\omega_{0}$

In [ ]:
# Basic w0 loop: one scalogram PDF per w0, no period-range parsing
# (equivalent to the original loop, w0_step=2.0).
output_dir = os.path.join(main_directory, "SCALOGRAM_LOOP")

sw.scalogram.w0_loop(
    wavelet, output_dir,
    w0_min=5.5, w0_max=196.0, w0_step=2.0,
    permin=min_period_analysis, permax=max_period_analysis, deltaj=0.05,
    percentile=percentile, planet_periods=p_planet,
    save_period_ranges=False,
)


In [ ]:
# Same loop, but also capturing wavepal's "Re-estimated period range" text
# for each w0 (equivalent to the original loop with w0_step=0.5, which wrote
# period_ranges.txt).
output_dir = os.path.join(main_directory, "SCALOGRAM_LOOP")

period_ranges = sw.scalogram.w0_loop(
    wavelet, output_dir,
    w0_min=5.5, w0_max=196.0, w0_step=0.5,
    permin=min_period_analysis, permax=max_period_analysis, deltaj=0.05,
    percentile=percentile, planet_periods=p_planet,
    save_period_ranges=True,
)
# period_ranges -> list[(w0, period_min, period_max)], also saved to
# output_dir/period_ranges.txt


## 3. FILTER:

In [ ]:
# From the dataset filename: Prot_055.0d and P1_030.250d
p_rot = 55.0
p_rot_half = p_rot / 2.0
p_planets = 30.25  # pass a list here (e.g. [30.25, 8.6]) if there is more than one planet


### 3.1. Simple filter (single w0 range, band around Prot)

In [ ]:
w0_grid = sw.make_w0_grid(5.5, 20.0, 0.5)

best_single = sw.single_filter_sweep(
    myt, myRV, myRV_err, w0_grid,
    bands=(p_rot - 0.5, p_rot + 0.5),
    p_rot=p_rot, p_rot_half=p_rot_half, p_planets=p_planets,
)
print(f"Best w0={best_single['w0']:.3f}  S_score={best_single['S_score']:.4g}")

sw.save_winner_file(
    myt, best_single["residuals"], myRV_err,
    os.path.join(main_directory, "single_filter_winner.dat"),
)


### 3.2. Double filter (bands around Prot and Prot/2, order 1 and/or 2)

In [ ]:
order1 = dict(
    w0_grid_1=sw.make_w0_grid(5.5, 7.0, 0.5),     # filters the Prot band first
    w0_grid_2=sw.make_w0_grid(14.5, 20.0, 0.25),  # then the Prot/2 band
    hw_1=0.5, hw_2=0.5,
)
order2 = dict(
    w0_grid_1=sw.make_w0_grid(11.5, 14.5, 0.1),   # filters the Prot/2 band first
    w0_grid_2=sw.make_w0_grid(5.5, 7.0, 0.1),     # then the Prot band
    hw_1=0.5, hw_2=0.5,
)

best_double = sw.run_double_filter_sweep(
    myt, myRV, myRV_err,
    prot_value=p_rot, prot_half_value=p_rot_half, planet_periods=p_planets,
    order1=order1, order2=order2,
)
print(f"Best order={best_double['order']} "
      f"w0_1={best_double['w0_1']:.3f} w0_2={best_double['w0_2']:.3f}  "
      f"S_score={best_double['S_score']:.4g}")

sw.save_winner_file(
    myt, best_double["residuals"], myRV_err,
    os.path.join(main_directory, "double_filter_winner.dat"),
)


### 3.3. CONAN fit on the winning residual

In [ ]:
planet_pars, n_planets = sw.build_planet_pars([
    dict(t0=3.2, t0_err=0.2, period=p_planets, period_err=0.01, k_prior_max=10.0),
])

# Requires CONAN installed (see spot_wave README) -> uncomment to run:
# sw.run_conan_fit(
#     filtered_data_files=["double_filter_winner.dat"],
#     data_path=main_directory,
#     output_folder=os.path.join(main_directory, "CONAN_OUT"),
#     planet_pars=planet_pars, m_star=(0.467, 0.02),
#     gamma_prior=[(0, 30)], n_planets=n_planets,
# )
# metrics = sw.extract_conan_metrics(os.path.join(main_directory, "CONAN_OUT"), n_data_points=len(myt))
# k_posteriors = sw.extract_all_k_posteriors(os.path.join(main_directory, "CONAN_OUT"), n_planets=n_planets)
